# Stage 4: Test Set Evaluation + CPU Latency Benchmarking

[1] Restoring Environment:

In [1]:
!pip install transformers scikit-learn pandas numpy torch -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch

# Load test set
test_df  = pd.read_csv('/content/drive/MyDrive/KD_Project/test.csv')

# Label maps
SENTIMENT_MAP = {"negative": 0, "neutral": 1, "positive": 2}
URGENCY_MAP   = {"non-urgent": 0, "urgent": 1}
INV_SENTIMENT = {v: k for k, v in SENTIMENT_MAP.items()}
INV_URGENCY   = {v: k for k, v in URGENCY_MAP.items()}

test_df["sentiment_id"] = test_df["sentiment"].map(SENTIMENT_MAP)

print(f"Test samples : {len(test_df)}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"\nTest class distribution:")
print(test_df["sentiment"].value_counts().to_string())

Mounted at /content/drive
Test samples : 485
GPU available: True

Test class distribution:
sentiment
neutral     288
positive    136
negative     61


[2] Rebuild Model Architecture + Load Weights:

In [2]:
import torch.nn as nn
from transformers import DistilBertModel, DistilBertTokenizerFast

class DualHeadDistilBERT(nn.Module):
    def __init__(self, num_sentiment=3, num_urgency=2, dropout=0.3):
        super(DualHeadDistilBERT, self).__init__()
        self.distilbert     = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout        = nn.Dropout(dropout)
        self.sentiment_head = nn.Linear(768, num_sentiment)
        self.urgency_head   = nn.Linear(768, num_urgency)

    def forward(self, input_ids, attention_mask):
        outputs    = self.distilbert(
            input_ids      = input_ids,
            attention_mask = attention_mask
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.sentiment_head(cls_output), self.urgency_head(cls_output)


WEIGHTS_PATH = '/content/drive/MyDrive/KD_Project/distilbert_finetuned/model_weights.pt'
TOKENIZER_PATH = '/content/drive/MyDrive/KD_Project/distilbert_finetuned'

# Load on GPU for test set evaluation
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model     = DualHeadDistilBERT().to(device)
tokenizer = DistilBertTokenizerFast.from_pretrained(TOKENIZER_PATH)

state_dict = torch.load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print(f"Model loaded on  : {device}")
print(f"Tokenizer loaded : {TOKENIZER_PATH}")
print(f"Model ready for evaluation.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on  : cuda
Tokenizer loaded : /content/drive/MyDrive/KD_Project/distilbert_finetuned
Model ready for evaluation.


[3] Full Test Set Evaluation (GPU):

In [3]:
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score

class TestDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids'      : self.encodings['input_ids'][idx],
            'attention_mask' : self.encodings['attention_mask'][idx],
            'label'          : self.labels[idx]
        }


test_dataset = TestDataset(
    texts     = test_df["text"].values,
    labels    = test_df["sentiment_id"].values,
    tokenizer = tokenizer
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

all_preds  = []
all_labels = []
all_urgency_preds = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        sentiment_logits, urgency_logits = model(input_ids, attention_mask)

        sentiment_preds = torch.argmax(sentiment_logits, dim=1).cpu().numpy()
        urgency_preds   = torch.argmax(urgency_logits,   dim=1).cpu().numpy()

        all_preds.extend(sentiment_preds)
        all_labels.extend(batch['label'].numpy())
        all_urgency_preds.extend(urgency_preds)

# Results
macro_f1    = f1_score(all_labels, all_preds, average='macro')
accuracy    = (np.array(all_preds) == np.array(all_labels)).mean()

print("=" * 55)
print("FINAL TEST SET RESULTS — SENTIMENT CLASSIFICATION")
print("=" * 55)
print(classification_report(
    all_labels, all_preds,
    target_names=["negative", "neutral", "positive"]
))
print(f"Macro F1 Score : {macro_f1:.4f}")
print(f"Accuracy       : {accuracy:.4f}")

# Urgency distribution on test set
from collections import Counter
urgency_counts = Counter(all_urgency_preds)
print(f"\nUrgency predictions on test set:")
print(f"  non-urgent : {urgency_counts[0]} ({urgency_counts[0]/len(all_urgency_preds)*100:.1f}%)")
print(f"  urgent     : {urgency_counts[1]} ({urgency_counts[1]/len(all_urgency_preds)*100:.1f}%)")

# Save results
results_dict = {
    "true_label"      : [INV_SENTIMENT[l] for l in all_labels],
    "predicted_label" : [INV_SENTIMENT[p] for p in all_preds],
    "urgency_pred"    : [INV_URGENCY[u] for u in all_urgency_preds],
    "correct"         : [l == p for l, p in zip(all_labels, all_preds)]
}
results_df = pd.DataFrame(results_dict)
results_df.to_csv('/content/drive/MyDrive/KD_Project/test_predictions.csv', index=False)
print(f"\nPredictions saved to Drive.")

FINAL TEST SET RESULTS — SENTIMENT CLASSIFICATION
              precision    recall  f1-score   support

    negative       0.70      0.80      0.75        61
     neutral       0.80      0.87      0.83       288
    positive       0.76      0.57      0.66       136

    accuracy                           0.78       485
   macro avg       0.75      0.75      0.75       485
weighted avg       0.78      0.78      0.77       485

Macro F1 Score : 0.7452
Accuracy       : 0.7773

Urgency predictions on test set:
  non-urgent : 472 (97.3%)
  urgent     : 13 (2.7%)

Predictions saved to Drive.


[4] Colab CPU Latency Benchmark:

In [4]:
import time
import tracemalloc

# Force CPU — simulate laptop deployment environment
cpu_device = torch.device('cpu')
model_cpu  = DualHeadDistilBERT()
model_cpu.load_state_dict(torch.load(WEIGHTS_PATH, map_location='cpu'))
model_cpu.eval()

print("Model loaded on CPU for latency benchmark.")
print("Running 100 inference passes on single samples...\n")

# Use 100 random test sentences
sample_texts = test_df["text"].sample(100, random_state=42).values
latencies    = []

for text in sample_texts:
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors  = 'pt',
        truncation      = True,
        padding         = True,
        max_length      = 128
    )

    # Measure inference time
    start = time.perf_counter()
    with torch.no_grad():
        sentiment_logits, urgency_logits = model_cpu(
            inputs['input_ids'],
            inputs['attention_mask']
        )
    end = time.perf_counter()

    latency_ms = (end - start) * 1000
    latencies.append(latency_ms)

latencies = np.array(latencies)

# Memory benchmark
tracemalloc.start()
inputs = tokenizer(
    sample_texts[0],
    return_tensors = 'pt',
    truncation     = True,
    padding        = True,
    max_length     = 128
)
with torch.no_grad():
    model_cpu(inputs['input_ids'], inputs['attention_mask'])
_, peak_memory = tracemalloc.get_traced_memory()
tracemalloc.stop()
peak_memory_mb = peak_memory / 1024 / 1024

# Model size on disk
import os
model_size_mb = os.path.getsize(WEIGHTS_PATH) / 1024 / 1024

print("=" * 55)
print("COLAB CPU LATENCY BENCHMARK")
print("=" * 55)
print(f"Hardware        : Google Colab CPU")
print(f"Samples tested  : 100")
print(f"Mean latency    : {latencies.mean():.2f} ms")
print(f"Median latency  : {np.median(latencies):.2f} ms")
print(f"Min latency     : {latencies.min():.2f} ms")
print(f"Max latency     : {latencies.max():.2f} ms")
print(f"Std deviation   : {latencies.std():.2f} ms")
print(f"Peak RAM usage  : {peak_memory_mb:.2f} MB")
print(f"Model size      : {model_size_mb:.2f} MB")
print(f"SLA target      : ≤ 50ms")
print(f"SLA status      : {'PASS' if latencies.mean() <= 50 else 'FAIL'}")

# Save benchmark results
latency_df = pd.DataFrame({
    "hardware"   : ["Colab CPU"] * 100,
    "latency_ms" : latencies
})
latency_df.to_csv(
    '/content/drive/MyDrive/KD_Project/latency_colab_cpu.csv',
    index=False
)
print(f"\nLatency results saved to Drive.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on CPU for latency benchmark.
Running 100 inference passes on single samples...

COLAB CPU LATENCY BENCHMARK
Hardware        : Google Colab CPU
Samples tested  : 100
Mean latency    : 79.40 ms
Median latency  : 74.28 ms
Min latency     : 48.77 ms
Max latency     : 148.16 ms
Std deviation   : 19.62 ms
Peak RAM usage  : 0.02 MB
Model size      : 253.20 MB
SLA target      : ≤ 50ms
SLA status      : FAIL

Latency results saved to Drive.


[5] Download Model to Laptop:

In [5]:
from google.colab import files
import shutil
import zipfile

# Zip the entire model folder
shutil.make_archive(
    '/content/distilbert_finetuned',
    'zip',
    '/content/drive/MyDrive/KD_Project/distilbert_finetuned'
)

# Download the zip
files.download('/content/distilbert_finetuned.zip')

print("Download started.")
print("Unzip on your laptop and run the local benchmark script.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.
Unzip on your laptop and run the local benchmark script.


[6] Save Complete Results Summary to Drive:

In [7]:
import os
import pandas as pd

# ── Real computed values from previous cells ──────────────────
model_size_mb = os.path.getsize(WEIGHTS_PATH) / 1024 / 1024

# ── Status evaluator ──────────────────────────────────────────
def get_status(value_str, target_str):
    try:
        val = float(value_str.replace("%", "").strip())
        if "≥" in target_str:
            tgt = float(target_str.replace("≥", "").replace("%", "").strip())
            if val >= tgt:
                return "✅ Pass"
            elif val >= tgt * 0.92:
                return "⚠️ Near target"
            else:
                return "❌ Below target"
        elif "≤" in target_str:
            tgt = float(
                target_str.replace("≤", "")
                           .replace("ms", "")
                           .replace("MB", "")
                           .strip()
            )
            return "✅ Pass" if val <= tgt else "❌ Above target"
        else:
            return "—"
    except:
        return "—"

# ── Build summary table with real values ─────────────────────
rows = [
    {
        "metric" : "Teacher Accuracy (Llama-3.1-8B)",
        "value"  : "0.8044",
        "target" : "≥ 0.88"
    },
    {
        "metric" : "Teacher Cohen Kappa",
        "value"  : "0.6362",
        "target" : "≥ 0.75"
    },
    {
        "metric" : "Student Val Macro F1",
        "value"  : "0.8028",
        "target" : "≥ 0.85"
    },
    {
        "metric" : "Student Test Macro F1",
        "value"  : f"{macro_f1:.4f}",
        "target" : "≥ 0.85"
    },
    {
        "metric" : "Student Test Accuracy",
        "value"  : f"{accuracy:.4f}",
        "target" : "≥ 0.88"
    },
    {
        "metric" : "Colab CPU Mean Latency (ms)",
        "value"  : f"{latencies.mean():.2f}",
        "target" : "≤ 50"
    },
    {
        "metric" : "Model Size (MB)",
        "value"  : f"{model_size_mb:.2f}",
        "target" : "≤ 300"
    },
    {
        "metric" : "Parameter Reduction vs Teacher",
        "value"  : "98.3",
        "target" : "≥ 98"
    },
    {
        "metric" : "Total Training Samples",
        "value"  : "3876",
        "target" : "—"
    },
    {
        "metric" : "Failed Pseudo-Labels",
        "value"  : "0",
        "target" : "—"
    },
]

# ── Add computed status to each row ──────────────────────────
for row in rows:
    row["status"] = get_status(row["value"], row["target"])

summary_df = pd.DataFrame(rows)

# ── Save to Drive ─────────────────────────────────────────────
summary_df.to_csv(
    '/content/drive/MyDrive/KD_Project/results_summary.csv',
    index=False
)

# ── Print ─────────────────────────────────────────────────────
print("=" * 65)
print("PROJECT RESULTS SUMMARY")
print("=" * 65)
print(summary_df.to_string(index=False))
print("=" * 65)
print("Results saved to Drive.")

PROJECT RESULTS SUMMARY
                         metric  value target         status
Teacher Accuracy (Llama-3.1-8B) 0.8044 ≥ 0.88 ❌ Below target
            Teacher Cohen Kappa 0.6362 ≥ 0.75 ❌ Below target
           Student Val Macro F1 0.8028 ≥ 0.85 ⚠️ Near target
          Student Test Macro F1 0.7452 ≥ 0.85 ❌ Below target
          Student Test Accuracy 0.7773 ≥ 0.88 ❌ Below target
    Colab CPU Mean Latency (ms)  79.40   ≤ 50 ❌ Above target
                Model Size (MB) 253.20  ≤ 300         ✅ Pass
 Parameter Reduction vs Teacher   98.3   ≥ 98         ✅ Pass
         Total Training Samples   3876      —              —
           Failed Pseudo-Labels      0      —              —
Results saved to Drive.
